### Developing the Evaluation / Visualization for Fixed Observation Rollouts

In [1]:
import os
import glob
import pickle
import imageio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

import robomimic
import robomimic.dev.hparam.utils.pkl_utils as pkl_utils
import robomimic.dev.hparam.utils.plot_utils as plot_utils
import robomimic.dev.hparam.metrics.smoothness_metrics as SmoothnessMetrics
import robomimic.dev.hparam.metrics.metric_fn as EvalMetrics
import robomimic.dev.hparam.utils.stat_utils as stat_utils
import robomimic.dev.hparam.utils.demo_utils as demo_utils
import robomimic.dev.hparam.metrics.metric_fn as Metrics
from robomimic.dev.hparam.eval_suite import DemoEvalSuite, HparamAnalyzer
from robomimic.utils import action_utils
import robomimic.dev.hparam.metrics.diversity_metrics as DiversityMetrics


In [2]:
# log_path = "/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 00:02:09.644066.pkl"
# log_path = "/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 00:44:09.066133.pkl"
# log_path = "/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 01:39:51.821603.pkl"
log_path = "/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 01:44:23.265916.pkl"
demo_idx = 180

In [3]:
demo = pkl_utils.load_demo_from_pkl(log_path, demo_idx=demo_idx)

In [ ]:
view_init = (10, 10)
figsize = (5, 5)
plot_utils.plot_predictions_3d(np.array(demo["preds"])[:, :, :3], colormap="Reds", view_init=view_init, new_fig=True, figsize=figsize)
plot_utils.plot_predictions_3d(np.array(demo["demo_preds"])[:, :16, :3], colormap="Greens", view_init=view_init, new_fig=False, figsize=figsize)


### Multimodality Visualization Video

In [20]:
# User can specify the video kwargs
view_init = "xz"
figsize = (5, 5)
xlim    = (-0.1, 0)
ylim    = (0, 0.2)
zlim    = (0.9, 1.1)
demo_pred_color = "viridis"
multi_pred_color = "Greens"

video_kwargs = dict(figsize=figsize, xlim=xlim, ylim=ylim, zlim=zlim, view_init=view_init)

In [ ]:
time_idx = 0
plot_utils.plot_predictions_3d(np.array(demo["multi_preds"])[time_idx, :, :, :3], 
                               colormap="Greens",
                               new_fig=True, 
                               **video_kwargs)

In [5]:
def generate_video(demos, view_init="xz", figsize=(6.4, 6.4), xlim=None, ylim=None, zlim=None,
                   demo_pred_color="viridis", multi_pred_color="Reds",
                   output_filename=None, fps=10):
    """
    Generates a 3D prediction video from demo data.

    Parameters:
    - demos (list of dicts): The demo data containing 'demo_preds' and 'multi_preds'.
    - view_init (str): Initial view angle for the plot. Default is "xz".
    - figsize (tuple): Size of the figure. Default is (5, 5).
    - xlim, ylim, zlim (tuple): Limits for the x, y, z axes respectively.
    - demo_pred_color (str): Color map for demo predictions. Default is "viridis".
    - multi_pred_color (str): Color map for multiple predictions. Default is "Reds".
    - output_filename (str): Filename for the output video. If None, defaults to "predictions_3d_{view_init}.mp4".
    - fps (int): Frames per second for the video. Default is 10.
    """
    if isinstance(demos, dict):
        demos = [demos]

    if isinstance(multi_pred_color, str):
        multi_pred_color = [multi_pred_color]

    points3D = []
    for demo in demos:
        points3D.append(np.array(demo["multi_preds"])[:, :, :, :3].reshape(-1, 3))
    points3D = np.concatenate(points3D, axis=0)
    
    if any(v is None for v in [xlim, ylim, zlim]):
        xlim = xlim or (points3D[:, 0].min(), points3D[:, 0].max())
        ylim = ylim or (points3D[:, 1].min(), points3D[:, 1].max())
        zlim = zlim or (points3D[:, 2].min(), points3D[:, 2].max())

    video_kwargs = dict(figsize=figsize, xlim=xlim, ylim=ylim, zlim=zlim, view_init=view_init)
    
    if output_filename is None:
        output_filename = f"predictions_3d_{view_init}.mp4"
    
    video_writer = imageio.get_writer(output_filename, fps=fps)
    
    # Draw the video
    demo_peek = demos[0]
    # plot_utils.plot_predictions_3d(np.array(demo_peek["demo_preds"])[0:1, :, :3], 
    #                                colormap=demo_pred_color,
    #                                new_fig=True, 
    #                                **video_kwargs)
    
    for i in range(len(demo_peek["demo_preds"])):
        result, fig = plot_utils.plot_predictions_3d(np.array(demo_peek["demo_preds"])[i:i+1, :, :3], 
                                            colormap=demo_pred_color, 
                                            new_fig=True,
                                            alpha=1.0,
                                            **video_kwargs)
        for idx, demo in enumerate(demos):
            plot_utils.plot_predictions_3d(np.array(demo["multi_preds"])[i, :, :, :3], 
                                                         colormap=multi_pred_color[idx],
                                                         new_fig=False,
                                                         alpha=0.5,
                                                         **video_kwargs)
        
        fig.canvas.draw()
        image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        video_writer.append_data(image)
    
    video_writer.close()

In [ ]:
results

In [ ]:
# Inpainting nres 1, nres 10

log_dir  = "/home/wjung85/Repo/projects/FastIL/logs/multimodality"
video_dir  = "/home/wjung85/Repo/projects/FastIL/videos/multimodality"
results  = pkl_utils.get_result_pkl_files(log_dir)
print(results)
demo_idx_vec = [180, 181, 182, 183, 184]
view_init_vec = ["xz", "xy", "yz"]

demos = [pkl_utils.load_demo_from_pkl(result, demo_idx=demo_idx) for result in results]
# demos = [demos[0], demos[1], demos[3]] # cfg_weight
# demos = [demos[0], demos[4], demos[-1]] # cfg_weight_vs_inpainting
demos = [demos[0], demos[7], demos[8]]
# demos = [demos[4], demos[5]] # cfg_weight_vs_inpainting


for view_init in view_init_vec:
    for demo_idx in demo_idx_vec:
        demos = [pkl_utils.load_demo_from_pkl(result, demo_idx=demo_idx) for result in results]
        demos = [demos[0], demos[7], demos[8]]
        video_path = os.path.join(video_dir, f"multimodality_video_demo_{demo_idx}_view_{view_init}.mp4")
        generate_video(demos, 
                    view_init=view_init, 
                    figsize=(6.4, 6.4), 
                    multi_pred_color=["Greens", "Reds", "Blues"], 
                    output_filename=video_path)



In [ ]:
from tqdm import tqdm
log_dir = "/home/wjung85/Repo/projects/FastIL/logs/multimodality"
video_dir = "/home/wjung85/Repo/projects/FastIL/videos/multimodality"

os.makedirs(video_dir, exist_ok=True)

demo_idx_vec = [180, 181, 182, 183, 184]
results = pkl_utils.get_result_pkl_files(log_dir)

for result in tqdm(results):
    for demo_idx in demo_idx_vec:
        demo = pkl_utils.load_demo_from_pkl(result, demo_idx=demo_idx)
        for view_init in ["xz", "xy", "yz"]:
            video_name = f"multimodality_video_{os.path.basename(result).split('.pkl')[0]}_{demo_idx}_{view_init}.mp4"
            video_path = os.path.join(video_dir, video_name)
            generate_video(demo, view_init=view_init, output_filename=video_path)

In [ ]:
os.path.basename(result)

In [ ]:
np.array(demo["multi_preds"])[:, :, :, :3].reshape(-1, 3).min(axis=0)

In [ ]:
log_paths = ["/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 01:44:23.265916.pkl",
             "/home/wjung85/Repo/projects/FastIL/logs/fixed_obs/square_image_cond_cfg_bl_iter_0_2025-01-24 01:39:51.821603.pkl"]
metric_dict = {}
for log_path in log_paths:
    metrics = []
    for demo_idx in [180, 181, 182, 183, 184]:
        demo = pkl_utils.load_demo_from_pkl(log_path, demo_idx=demo_idx)
        for i in range(len(demo["multi_preds"])):
            trajectory = np.array(demo["multi_preds"][i])[:, :, :3]
            metric = DiversityMetrics.compute_dtw_distance(trajectory)
            metrics.append(metric)
    
    metric_dict[log_path] = metrics